# Otimização de Seções — Treliça 2D de Duas Barras

Tradução em Python dos scripts MATLAB da pasta `Ex1`
(`Ex1_trelica_2D.m`, `massa_trelica.m`, `tensao_maxima.m`).

Dimensionamento das áreas de seção transversal $A_1, A_2$ de duas barras de comprimento
unitário, de forma a **minimizar a massa** da treliça sujeita a uma força de compressão
$F_c$ em cada barra, respeitando a tensão admissível $\sigma_a$ do material:
$$\min_{A_1,A_2}\; \rho L (A_1+A_2) \quad \text{s.a.} \quad \dfrac{F_c}{A_i}\le \sigma_a,\;\; A_i\ge A_{min}$$

O problema é resolvido numericamente (equivalente ao `fmincon`/SQP do MATLAB) e também
de forma analítica, encontrando as raízes do polinômio que caracteriza o ponto ótimo.

In [1]:
import numpy as np
from scipy.optimize import minimize

## Função objetivo e restrição (`massa_trelica.m`, `tensao_maxima.m`)

A restrição de tensão do MATLAB é escrita na forma $g(A)=F_c/A-\sigma_a\le 0$; aqui ela é
reescrita de forma equivalente, porém numericamente mais estável para o otimizador (evita
a curvatura acentuada de $1/A$ perto de $A=0$): $g(A)=\sigma_a A - F_c \ge 0$.

In [2]:
rho = 2700       # kg/m^3
comprimento = 1  # m
Fc = 100e4       # N (força de compressão em cada barra)
sigma_a = 100e6  # Pa (tensão admissível)

Amin = [1e-5, 1e-5]


def massa_trelica(A):
    return rho * comprimento * (A[0] + A[1])


def tensao_maxima(A):
    """g(A) >= 0 <=> Fc/A <= sigma_a (forma reformulada, numericamente estável)."""
    return np.array([sigma_a * A[0] - Fc, sigma_a * A[1] - Fc])

## Otimização (`Ex1_trelica_2D.m`)

Chute inicial $A=[0.01,\,0.01]$ m². Como a solução ótima ocorre exatamente no limite da
restrição de tensão, o chute inicial já está muito próximo do ótimo — o histórico de
convergência do solver pode reportar passo nulo, o que é esperado.

In [3]:
A0 = [0.01, 0.01]

res = minimize(
    massa_trelica, A0, method='SLSQP',
    bounds=[(Amin[0], None), (Amin[1], None)],
    constraints={'type': 'ineq', 'fun': tensao_maxima},
    options={'maxiter': 200, 'ftol': 1e-12},
)

print('Convergiu:', res.success, '-', res.message)
print('A =', res.x)
print('massa =', res.fun)

Convergiu: True - Optimization terminated successfully
A = [0.01 0.01]
massa = 54.0


## Verificação analítica — raízes do polinômio

O script original também resolve o problema de forma fechada: impondo a restrição de área
mínima ativa em uma das barras ($A_2=A_{min}$) e a tensão admissível ativa na outra, chega-se
a um polinômio de 4º grau em $A$,
$$A^4 - A_{min}A^3 + \sigma_a F_c\,A - F_c^2 = 0,$$
cujas raízes são calculadas para comparação com o resultado numérico.

In [4]:
p = [1, -Amin[0], 0, sigma_a * Fc, -Fc * Fc]
A1_roots = np.roots(p)
A1_roots

array([-4.64158917e+04    +0.j        ,  2.32079408e+04+40197.33843831j,
        2.32079408e+04-40197.33843831j,  1.00000000e-02    +0.j        ])